In [1]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")




/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
class BlindnessDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, test=False):
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        self.test = test

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        img_name = self.annotations.iloc[idx, 0] + ".png"
        img_path = os.path.join(self.root_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        if self.test:
            return image
        else:
            label = int(self.annotations.iloc[idx, 1])
            return image, label




In [3]:
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)



In [4]:
# Test data
test_csv_file = "/kaggle/input/aptos2019-blindness-detection/test.csv"
test_root_dir = "/kaggle/input/aptos2019-blindness-detection/test_images"
test_dataset = BlindnessDataset(
    test_csv_file, test_root_dir, transform=transform, test=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)

# Train/validation split
train_csv_file = "/kaggle/input/aptos2019-blindness-detection/train.csv"
train_root_dir = "/kaggle/input/aptos2019-blindness-detection/train_images"
full_train_dataset = BlindnessDataset(
    train_csv_file, train_root_dir, transform=transform, test=False
)

train_idx, val_idx = train_test_split(
    np.arange(len(full_train_dataset)),
    test_size=0.2,
    stratify=full_train_dataset.annotations["diagnosis"],
    random_state=42,
)

train_dataset = torch.utils.data.Subset(full_train_dataset, train_idx)
val_dataset = torch.utils.data.Subset(full_train_dataset, val_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)



In [5]:
model_names_to_load = ["resnet18", "seresnext50_32x4d", "seresnext101_32x4d"]
models_list = []
loaded_model_names = []

for name in model_names_to_load:
    try:
        model = timm.create_model(name, pretrained=True, num_classes=5)
        model.to(device)
        model.train()
        models_list.append(model)
        loaded_model_names.append(name)
    except Exception as e:
        print(f"Warning: could not load model {name}: {e}")

if not loaded_model_names:
    raise RuntimeError("No models were successfully loaded.")



model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/111M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/196M [00:00<?, ?B/s]

In [6]:
criterion = nn.CrossEntropyLoss()
optimizers = [torch.optim.Adam(m.parameters(), lr=1e-4) for m in models_list]
epochs = 2  # lightweight training

for epoch in range(epochs):
    for model, optimizer in zip(models_list, optimizers):
        model.train()
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} training"):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    print(f"Completed epoch {epoch+1}")

# set models to eval mode for validation/inference
for model in models_list:
    model.eval()



Epoch 1 training: 100%|██████████| 83/83 [00:55<00:00,  1.49it/s]


Completed epoch 1


Epoch 2 training: 100%|██████████| 83/83 [00:56<00:00,  1.48it/s]

Completed epoch 2


In [7]:
validation_scores = {}
for name, model in zip(loaded_model_names, models_list):
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Validating {name}"):
            images = images.to(device, non_blocking=True)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu()
            all_preds.append(preds)
            all_targets.append(labels.cpu())
    preds_cat = torch.cat(all_preds).numpy()
    targets_cat = torch.cat(all_targets).numpy()
    kappa = cohen_kappa_score(targets_cat, preds_cat, weights="quadratic")
    validation_scores[name] = kappa
    print(f"Model {name} validation QWK: {kappa:.4f}")



Validating resnet18: 100%|██████████| 21/21 [00:16<00:00,  1.25it/s]


Model resnet18 validation QWK: 0.7167


Validating seresnext50_32x4d: 100%|██████████| 21/21 [00:16<00:00,  1.28it/s]


Model seresnext50_32x4d validation QWK: 0.8484


Validating seresnext101_32x4d: 100%|██████████| 21/21 [00:16<00:00,  1.31it/s]

Model seresnext101_32x4d validation QWK: 0.8742


In [8]:
total_score = sum(validation_scores.get(k, 0) for k in loaded_model_names)
if total_score == 0:
    weights = {k: 1.0 / len(loaded_model_names) for k in loaded_model_names}
else:
    weights = {k: validation_scores.get(k, 0) / total_score for k in loaded_model_names}



In [9]:
all_outputs = []
with torch.no_grad():
    for images in tqdm(test_loader, desc="Inference"):
        images = images.to(device, non_blocking=True)
        weighted_preds = []
        for name, model in zip(loaded_model_names, models_list):
            w = weights.get(name, 1.0 / len(models_list))
            preds = nn.functional.softmax(model(images), dim=1) * w
            weighted_preds.append(preds)
        ensemble_output = torch.stack(weighted_preds).sum(dim=0)  # (batch, 5)
        all_outputs.extend(ensemble_output.cpu().numpy())

all_outputs = np.array(all_outputs)
final_predictions = np.argmax(all_outputs, axis=1)



Inference: 100%|██████████| 23/23 [00:09<00:00,  2.34it/s]


In [10]:
submission_df = pd.DataFrame(
    {"id_code": pd.read_csv(test_csv_file)["id_code"], "diagnosis": final_predictions}
)
submission_df.to_csv("submission.csv", index=False)